In [2]:
import requests
from bs4 import BeautifulSoup
import json
import random
import pandas as pd

# Base

In [3]:
import requests
from bs4 import BeautifulSoup
import json
import random

class UserAgentRotator:
    _COMMON_UA_URL = "https://www.useragents.me/"

    _STATIC_UA_POOL = [
        "Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1; SV1; .NET CLR 1.1.4322)",
        "Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko",
        "Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1)",

        # Chrome
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 11_6_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",

        #Firefox
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 12_4) Gecko/20100101 Firefox/114.0",
        "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:113.0) Gecko/20100101 Firefox/113.0",
    ]
    def __init__(self):
        self.update()


    def _fetch_ua(self):

        r = requests.get(self._COMMON_UA_URL)
        soup = BeautifulSoup(r.text, "html.parser")

        ua_div = soup.find("div", attrs={"id": "most-common-desktop-useragents-json-csv"})
        ua = ua_div.find("textarea", class_="form-control")
        ua_list = json.loads(ua.text.strip())
        ua_weight = [entry["pct"] for entry in ua_list]
        ua_list = [entry["ua"] for entry in ua_list]

        return ua_list, ua_weight
    

    def update(self):
        """
        Fetch COMMON UA URL and load UA list to memory
        """
        self.common_ua_list, self.common_ua_weight = self._fetch_ua()

    
    def rotate(self, static_common=[30, 70]):
        use_static_pool = random.choices([True, False], weights=static_common, k=1)[0]
        if use_static_pool:
            return random.choice(self._STATIC_UA_POOL)
        else:
            return random.choices(self.common_ua_list, weights=self.common_ua_weight, k=1)[0]


In [4]:
from abc import ABC, abstractmethod
import logging
import os

class BaseCrawler(ABC):

    def __init__(self, log_level="INFO"):
        self.ua_rotator = UserAgentRotator()
        # self.proxy_manager = ProxyManager()
        self.logger = logging.getLogger(self.__class__.__name__)
        self.setup_logger(log_level)

    
    @abstractmethod
    def crawl(self):
        raise NotImplementedError(f"{self.__class__.__name__} must have crawl method")       
    

    def setup_logger(self, log_level="INFO"):
        log_level = getattr(logging, log_level, logging.INFO)
        self.logger.setLevel(logging.DEBUG)

        console_handler = logging.StreamHandler()
        console_handler.setLevel(log_level)

        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        console_handler.setFormatter(formatter)

        if not self.logger.handlers:
            self.logger.addHandler(console_handler)
            self.logger.propagate = False # prevent duplicate log
        


## Traffic img

In [5]:
import pandas as pd
import json
import os
import re
import requests

class TrafficCrawler(BaseCrawler):
    _BASE_URL = "https://giaothong.hochiminhcity.gov.vn"

    def __init__(self, log_level="INFO"):
        super().__init__(log_level)
        self._cam_image_url = os.path.join(self._BASE_URL, "render/ImageHandler.ashx")
        self._cam_info_url = os.path.join(self._BASE_URL, "ajaxpro/VDMS.Web.Library.AJAX.FolderAjax,VDMS.Web.Library.ashx")
    

    def get_cam_info(self, page: int = 1, limit: int = -1) -> list[dict]:
        with requests.Session() as session:
            payload = {
                "path": "/root/vdms/tangthu/data/layerdata/camera",
                "isInTree": True,
                "searchKey": "",
                "layer": ["CAMERA"],
                "detail": True,
                "page": page,
                "limit": limit,
                "filterQuery": ["Publish:true AND CamStatus:UP"],
                "sortby": {"SortInfo": [
                        {
                            "Field": "ModifiedDate",
                            "Direction": 1
                        }
                    ]},
                "returnFields": ["DisplayName", "CamId", "Disctrict"]
            }

            headers = {
                "Accept": "*/*",
                "Accept-Language": "en-US,en;q=0.9,vi;q=0.8",
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
                "Content-Type": "application/json; charset=UTF-8",
                "Origin": "https://giaothong.hochiminhcity.gov.vn",
                "Pragma": "no-cache",
                "Referer": "https://giaothong.hochiminhcity.gov.vn/",
                "sec-ch-ua-mobile": "?0",
                # "sec-ch-ua-platform": "Windows",
                "Sec-Fetch-Dest": "empty",
                "Sec-Fetch-Mode": "cors",
                "Sec-Fetch-Site": "same-origin",
                "User-Agent": self.ua_rotator.rotate(),
                "X-AjaxPro-Method": "SearchQuery"
            }

            with requests.Session() as session:
                _ = session.get(self._BASE_URL) # warmup to get cookies
                response = session.request("POST", self._cam_info_url, json=payload, headers=headers)
                with open("t.txt", "w") as f:
                    f.write(response.text)
                data = self._extract_cam_info(response.text)
                return data

    
    def _extract_cam_info(self, text) -> list[dict]:
        pattern = r'\[\{"__type":"VDMS\.Sense\.Helper\.Model\.FileProperty, VDMS\.Sense\.Helper\.Model.*?"Format":null\}\]'

        result = []
        matches = re.findall(pattern, text)
        for match in matches:
            json_data = json.loads(match)
            template = {data['Name']: data['Value'] for data in json_data}
            result.append(template)
        return result
    
    
    def crawl(self, id: str) -> bytes:
        headers = {
            "User-Agent": self.ua_rotator.rotate(),
            # "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8"
        }
        params = {
            "id": id
        }
        response = requests.get(self._cam_image_url, headers=headers, params=params)
        
        # if response.status_code == 200:
        return response.content

## weather

In [ ]:
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import unicodedata
import requests
import os
import re

class WeatherCrawler(BaseCrawler):
    _BASE_URL = "https://www.accuweather.com"

    def __init__(self):
        super().__init__()
        self._url_cities = os.path.join(self._BASE_URL, "vi/browse-locations/asi/vn")

        self._headers = {
            "Referer": "https://www.accuweather.com"
        }

    def crawl(self, url: str):
        result = {}
        soup = BeautifulSoup(requests.get(url, headers=self._headers).text, "html.parser")
        current_weather = soup.find("div", class_="current-weather-card card-module content-module")
        current_time = current_weather.find("p", class_="sub").text
        status = current_weather.find("div", class_="phrase").text # Sunny
        temp_c = current_weather.find("div", class_="display-temp").text
        properties = current_weather.find_all("div", class_="detail-item spaced-content")

        result.update({
            "current_time": current_time,
            "status": status,
            "temp_c": temp_c
        })

        for prop in properties:
            detail = prop.text.split("\n")
            detail = [_ for _ in detail if _ != ""]
            result.update({detail[0]: detail[1]})
        result

    def get_city_list(self) -> list[dict]:
        self._headers.update({"User-Agent": self.ua_rotator.rotate()})
    
        url = os.path.join(self._BASE_URL, "vi/browse-locations/asi/vn")
        headers = {
            "User-Agent": "insomnia/11.0.2",
            "Referer": "https://www.accuweather.com"
        }
        with requests.Session() as s:
            r = s.get(url, headers=headers)

        soup = BeautifulSoup(r.text, "html.parser")
        search_result = [tag for tag in soup.find_all("a", class_="search-result") if tag.get("class") == ["search-result"]]
        cities = [{"city": tag.text, "url": self._BASE_URL + tag["href"]} for tag in search_result]
        result = []
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = [executor.submit(self._get_district, city, headers) for city in cities]
            
            for future in as_completed(futures):
                try:
                    result.extend(future.result())
                except Exception as e:
                    self.logger.error(f"Error: {e}")

        return result

    def _preprocess(self, district: dict):
        # Process name
        d_name = district["district"].lower().replace(" ", "-").replace("quận", "district")
        d_name = unicodedata.normalize('NFD', d_name)
        d_name = ''.join(c for c in d_name if unicodedata.category(c) != 'Mn')
        d_name = d_name.replace('đ', 'd').replace('Đ', 'D')

        # Extract key
        match = re.search(r'key=(\d+)', district["district_url"])
        if match:
            key = match.group(1)
        
        return {"district_url": f"https://www.accuweather.com/vi/vn/{d_name}/{key}/weather-forecast/{key}", "district": district['district']}
    

    def _get_district(self, city, headers):
        with requests.Session() as s:
            response = s.get(city['url'], headers=headers)
            soup = BeautifulSoup(response.text, "html.parser")
            districts = [self._preprocess({"district": tag.text, "district_url": tag.get("href")}) for tag in soup.find_all("a", class_="search-result") if tag.get("class") == ["search-result"]]
            districts = [{**district, **city} for district in districts]
            return districts


In [10]:
crawler = WeatherCrawler()
a = crawler.get_city_list()
df = pd.DataFrame(a)
df.to_csv("temp_weather.tsv", sep="\t", index=True, header=False)

In [19]:
df.to_csv("temp_weather.tsv", sep="\t", index=False, header=True)

In [ ]:
from bs4 import BeautifulSoup
import requests
headers = {
    "User-Agent": "insomnia/11.0.2",
    "Referer": "https://www.accuweather.com"
}
url = "https://www.accuweather.com/en/vn/district-1/3554433/current-weather/3554433"

with requests.Session() as session:
    r = session.get(url, headers=headers)

result = {}
soup = BeautifulSoup(r.text, "html.parser")
current_weather = soup.find("div", class_="current-weather-card card-module content-module")
current_time = current_weather.find("p", class_="sub").text
status = current_weather.find("div", class_="phrase").text # Sunny
temp_c = current_weather.find("div", class_="display-temp").text
properties = current_weather.find_all("div", class_="detail-item spaced-content")

result.update({
    "current_time": current_time,
    "status": status,
    "temp_c": temp_c
})

for prop in properties:
    detail = prop.text.split("\n")
    detail = [_ for _ in detail if _ != ""]
    result.update({detail[0]: detail[1]})
result

In [38]:
result = {}
soup = BeautifulSoup(r.text, "html.parser")
current_weather = soup.find("div", class_="current-weather-card card-module content-module")
current_time = current_weather.find("p", class_="sub").text
status = current_weather.find("div", class_="phrase").text # Sunny
temp_c = current_weather.find("div", class_="display-temp").text
properties = current_weather.find_all("div", class_="detail-item spaced-content")

result.update({
    "current_time": current_time,
    "status": status,
    "temp_c": temp_c
})

for prop in properties:
    detail = prop.text.split("\n")
    detail = [_ for _ in detail if _ != ""]
    result.update({detail[0]: detail[1]})
result

{'current_time': '4:33 PM',
 'status': 'Sunny',
 'temp_c': '35°C\n',
 'RealFeel®': '39°',
 'RealFeel Shade™': '38°',
 'Max UV Index': '3 Moderate',
 'Wind': 'S 9 km/h',
 'Wind Gusts': '9 km/h',
 'Humidity': '46%',
 'Indoor Humidity': '46% (Extremely Humid)',
 'Dew Point': '22° C',
 'Pressure': '↔ 1006 mb',
 'Cloud Cover': '10%',
 'Visibility': '16 km',
 'Cloud Ceiling': '12200 m'}

In [13]:
headers = {
    "User-Agent": "insomnia/11.0.2",
    "Referer": "https://www.accuweather.com"
}
requests.get("https://www.accuweather.com/vi/vn/district-1/3554433/current-weather/3554433", headers=headers).text

'\n\n<!DOCTYPE html>\n<html lang="vi" class="accuweather">\n\n<head>\n\t<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\n\t\n\t<meta charset="utf-8" />\n\t<link rel="canonical" href="https://www.accuweather.com/vi/vn/district-1/3554433/current-weather/3554433" />\n\t<title>Th&#x1EDD;i ti&#x1EBF;t hi&#x1EC7;n t&#x1EA1;i &#x1EDF; Qu&#x1EAD;n 1, H&#x1ED3; Ch&#xED; Minh, Vi&#x1EC7;t Nam | AccuWeather</title>\n\t<meta name="Description" content="H&#xE3;y chu&#x1EA9;n b&#x1ECB; k&#x1EF9; cho ng&#xE0;y. H&#xE3;y ki&#x1EC3;m tra t&#xEC;nh tr&#x1EA1;ng hi&#x1EC7;n t&#x1EA1;i cho Qu&#x1EAD;n 1, H&#x1ED3; Ch&#xED; Minh, Vi&#x1EC7;t Nam cho ng&#xE0;y s&#x1EAF;p t&#x1EDB;i, c&#xF3; th&#xF4;ng tin radar, d&#x1EF1; b&#xE1;o theo gi&#x1EDD; v&#xE0; chi ti&#x1EBF;t &#x111;&#x1EBF;n m&#x1EE9;c theo ph&#xFA;t. ">\n\t<meta name="viewport" content="width=device-width, initial-scale=1.0" />\n\t<meta name="referrer" content="origin">\n\t\n\t\n\n\t<meta property="fb:profile_id" content="AccuWea

In [10]:
soup.prettify()

'<html>\n <head>\n  <title>\n   Access Denied\n  </title>\n </head>\n <body>\n  <h1>\n   Access Denied\n  </h1>\n  You don\'t have permission to access "http://www.accuweather.com/vi/vn/district-1/3554433/current-weather/3554433" on this server.\n  <p>\n   Reference #18.82e83217.1744447668.24ad3cb0\n   <p>\n    https://errors.edgesuite.net/18.82e83217.1744447668.24ad3cb0\n   </p>\n  </p>\n </body>\n</html>\n'